# Setup

In [13]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if 'data' in str(Path.cwd()) else Path.cwd()
SRC_DIR = REPO_ROOT / 'src'
DATA_DIR = Path(REPO_ROOT,'data/wellness_centre')  # Your CSV files

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"Repository: {REPO_ROOT}")
print(f"Source: {SRC_DIR}")
print(f"Data: {DATA_DIR}")

Repository: /home/jovyan/work/ERP/emt
Source: /home/jovyan/work/ERP/emt/src
Data: /home/jovyan/work/ERP/emt/data/wellness_centre


# Import orchestrator

In [2]:
# Cell 2: Import orchestrator
from orchestration import MigrationOrchestrator

orchestrator = MigrationOrchestrator(DATA_DIR)

In [3]:
# Cell 3: Check what data is available
summary = orchestrator.loader.get_summary()
print("Available data:")
for source, count in summary.items():
    print(f"  {source}: {count} records")

Available data:
  events: 25 records
  rooms: 54 records
  eggs: 103 records


In [4]:
# Cell 4: Run SMALL test first (10 records)
print("Running small test batch...")
test_results = orchestrator.process_batch(limit=10)

# Show what happened
report = orchestrator.generate_report(test_results)
print(report)

Running small test batch...
Starting batch processing (limit: 10)...

1. Loading CSV data...
   Loaded 10 events
   Loaded 10 room bookings
   Loaded 10 egg sales

2. Generating invoices...
   Generated 10 event invoices
   Generated 10 room invoices
   Generated 10 egg invoices
   Total: 30 invoices

3. Calculating totals...
   Event revenue: KES 597,400.00
   Room revenue: KES 156,600.00
   Egg revenue: KES 14,950.00
   Total revenue: KES 768,950.00
WELLNESS CENTRE MIGRATION REPORT

DATA LOADED:
  Events:          10
  Room Bookings:   10
  Egg Sales:       10

INVOICES GENERATED:
  Event Invoices:   10
  Room Invoices:    10
  Egg Invoices:     10
  Total:            30

REVENUE BREAKDOWN:
  Events:
    Subtotal: KES 515,000.00
    Tax:      KES 82,400.00
    Total:    KES 597,400.00

  Rooms:
    Subtotal: KES 135,000.00
    Tax:      KES 21,600.00
    Total:    KES 156,600.00

  Eggs:
    Subtotal: KES 14,950.00
    Tax:      KES 0.00
    Total:    KES 14,950.00

COMBINED TOTALS:


In [5]:
# Cell 5: Run FULL migration (remove limit!)
print("Running FULL migration...")
results = orchestrator.process_batch()  # ← No limit = ALL records

# Generate report
report = orchestrator.generate_report(results)
print(report)

Running FULL migration...
Starting batch processing (limit: ALL)...

1. Loading CSV data...
   Loaded 25 events
   Loaded 54 room bookings
   Loaded 103 egg sales

2. Generating invoices...
   Generated 25 event invoices
   Generated 54 room invoices
   Generated 103 egg invoices
   Total: 182 invoices

3. Calculating totals...
   Event revenue: KES 1,969,680.00
   Room revenue: KES 718,040.00
   Egg revenue: KES 136,840.00
   Total revenue: KES 2,824,560.00
WELLNESS CENTRE MIGRATION REPORT

DATA LOADED:
  Events:          25
  Room Bookings:   54
  Egg Sales:      103

INVOICES GENERATED:
  Event Invoices:   25
  Room Invoices:    54
  Egg Invoices:    103
  Total:           182

REVENUE BREAKDOWN:
  Events:
    Subtotal: KES 1,698,000.00
    Tax:      KES 271,680.00
    Total:    KES 1,969,680.00

  Rooms:
    Subtotal: KES 619,000.00
    Tax:      KES 99,040.00
    Total:    KES 718,040.00

  Eggs:
    Subtotal: KES 136,840.00
    Tax:      KES 0.00
    Total:    KES 136,840.00

COM

In [6]:
# Cell 6: Export ERPNext payloads
output_dir = Path(REPO_ROOT,'user-data/outputs')
export_info = orchestrator.export_erpnext_payloads(results, output_dir)

print(f"\nExported to: {export_info['file']}")
print(f"Invoice count: {export_info['count']}")


Exported to: /home/jovyan/work/ERP/emt/user-data/outputs/sales_invoices.json
Invoice count: 182


In [7]:
# Cell 7: Inspect a sample invoice
sample_invoice = results['invoices']['event_invoices'][0]

print("Sample Event Invoice:")
print(f"  Customer: {sample_invoice.customer}")
print(f"  Date: {sample_invoice.posting_date}")
print(f"  Items: {len(sample_invoice.items)}")
print(f"  Subtotal: {sample_invoice.subtotal}")
print(f"  Tax: {sample_invoice.total_tax}")
print(f"  Total: {sample_invoice.grand_total}")

# See ERPNext format
import json
payload = sample_invoice.to_erpnext_format()
print("\nERPNext JSON:")
print(json.dumps(payload, indent=2, default=str))

Sample Event Invoice:
  Customer: Dorothy Barasa
  Date: 2024-03-16
  Items: 1
  Subtotal: KES 35,000.00
  Tax: KES 5,600.00
  Total: KES 40,600.00

ERPNext JSON:
{
  "doctype": "Sales Invoice",
  "customer": "Dorothy Barasa",
  "customer_name": "Dorothy Barasa",
  "posting_date": "2024-03-16",
  "due_date": "2024-03-31",
  "items": [
    {
      "item_name": "Event Venue Hire - Dorothy's Nephew's 10th Birthday",
      "description": "Event Venue Hire - Dorothy's Nephew's 10th Birthday",
      "qty": 1.0,
      "rate": 35000.0,
      "amount": 35000.0,
      "uom": "Nos",
      "item_code": "Event Venue Hire"
    }
  ],
  "taxes": [
    {
      "charge_type": "On Net Total",
      "account_head": "VAT - WC",
      "description": "VAT @ 16%",
      "rate": 16.0,
      "tax_amount": 5600.0
    }
  ],
  "remarks": "Birthday Party event: Dorothy's Nephew's 10th Birthday (40 guests)"
}


In [9]:
from orchestration import MigrationOrchestrator
from pathlib import Path

orchestrator = MigrationOrchestrator(DATA_DIR)
results = orchestrator.process_batch(limit=10)

# Check first invoice
sample = results['invoices']['event_invoices'][0]
print(f"Customer: {sample.customer}")  # Should show real name now!

Starting batch processing (limit: 10)...

1. Loading CSV data...
   Loaded 10 events
   Loaded 10 room bookings
   Loaded 10 egg sales

2. Generating invoices...
   Generated 10 event invoices
   Generated 10 room invoices
   Generated 10 egg invoices
   Total: 30 invoices

3. Calculating totals...
   Event revenue: KES 597,400.00
   Room revenue: KES 156,600.00
   Egg revenue: KES 14,950.00
   Total revenue: KES 768,950.00
Customer: Dorothy Barasa


In [10]:
# Cell: Run FULL migration (all records)
print("Running FULL migration - ALL records...")
full_results = orchestrator.process_batch()  # ← No limit!

# Generate report
full_report = orchestrator.generate_report(full_results)
print(full_report)

Running FULL migration - ALL records...
Starting batch processing (limit: ALL)...

1. Loading CSV data...
   Loaded 25 events
   Loaded 54 room bookings
   Loaded 103 egg sales

2. Generating invoices...
   Generated 25 event invoices
   Generated 54 room invoices
   Generated 103 egg invoices
   Total: 182 invoices

3. Calculating totals...
   Event revenue: KES 1,969,680.00
   Room revenue: KES 718,040.00
   Egg revenue: KES 136,840.00
   Total revenue: KES 2,824,560.00
WELLNESS CENTRE MIGRATION REPORT

DATA LOADED:
  Events:          25
  Room Bookings:   54
  Egg Sales:      103

INVOICES GENERATED:
  Event Invoices:   25
  Room Invoices:    54
  Egg Invoices:    103
  Total:           182

REVENUE BREAKDOWN:
  Events:
    Subtotal: KES 1,698,000.00
    Tax:      KES 271,680.00
    Total:    KES 1,969,680.00

  Rooms:
    Subtotal: KES 619,000.00
    Tax:      KES 99,040.00
    Total:    KES 718,040.00

  Eggs:
    Subtotal: KES 136,840.00
    Tax:      KES 0.00
    Total:    KES 1

# Quick connection test

In [14]:
import os
import csv

# ─────────────────────────────────────────────────────────────
# EDIT THESE VALUES
# ─────────────────────────────────────────────────────────────

# Internal Docker URL — avoids Cloudflare round-trip latency.
# Fall back to external domain only if internal URL fails.
ERPNEXT_URL    = "http://erpnext-frontend:8080"
ERPNEXT_DOMAIN = "well.rosslyn.cloud"  # used as Host header for internal routing

# Company name exactly as entered in ERPNext setup wizard
COMPANY = "Wellness Centre"

# Path to API credentials file
# CSV format: two columns named 'api_key' and 'api_secret', one row of values
API_KEYS_FILE = "/home/jovyan/work/ERP/Wellness Centre/frappe_api_keys.csv"

# ─────────────────────────────────────────────────────────────
# Credential loading — CSV → environment variable → empty
# ─────────────────────────────────────────────────────────────
API_KEY    = ""
API_SECRET = ""

if os.path.exists(API_KEYS_FILE):
    with open(API_KEYS_FILE, newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            API_KEY    = row.get("api_key", "").strip()
            API_SECRET = row.get("api_secret", "").strip()
            break  # only read the first data row
    print(f"✓ Credentials loaded from {API_KEYS_FILE}")
else:
    # Fall back to environment variables
    API_KEY    = os.environ.get("ERPNEXT_API_KEY", "")
    API_SECRET = os.environ.get("ERPNEXT_API_SECRET", "")
    if API_KEY:
        print("✓ Credentials loaded from environment variables")
    else:
        print(f"⚠  Credentials file not found: {API_KEYS_FILE}")
        print("   Set ERPNEXT_API_KEY / ERPNEXT_API_SECRET environment variables,")
        print("   or create the CSV file, or paste keys directly into API_KEY / API_SECRET above.")

# ─────────────────────────────────────────────────────────────
# Validation
# ─────────────────────────────────────────────────────────────
problems = []
if not API_KEY:
    problems.append("API_KEY is empty")
if not API_SECRET:
    problems.append("API_SECRET is empty")
if not os.path.exists(DATA_DIR):
    problems.append(f"DATA_DIR not found: {DATA_DIR}")

if problems:
    print("\n⚠  Configuration problems:")
    for p in problems:
        print(f"   • {p}")
else:
    print(f"\n✓ Configuration looks good")
    print(f"  URL:      {ERPNEXT_URL}")
    print(f"  Domain:   {ERPNEXT_DOMAIN}")
    print(f"  Company:  {COMPANY}")
    print(f"  Data dir: {DATA_DIR}")
    print(f"  API key:  {'set (' + API_KEY[:6] + '...)' if API_KEY else 'MISSING'}")

✓ Credentials loaded from /home/jovyan/work/ERP/Wellness Centre/frappe_api_keys.csv

✓ Configuration looks good
  URL:      http://erpnext-frontend:8080
  Domain:   well.rosslyn.cloud
  Company:  Wellness Centre
  Data dir: /home/jovyan/work/ERP/emt/data/wellness_centre
  API key:  set (c7edf8...)


In [22]:
ERPNEXT_URL

'http://erpnext-frontend:8080'